# 01 - Download BCB SGS data

Download all manually verified Banco Central do Brasil SGS series listed in `data/series_dictionary.csv`.

In [1]:
import sys
print(sys.executable)

import pandas as pd
print(pd.__version__)

c:\Users\chico\brazil-directed-credit-monetary-policy\.venv-1\Scripts\python.exe
2.3.3


In [2]:
from datetime import date
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bcb_api import fetch_sgs_series, save_series_csv

In [ ]:
START_DATE = "2007-01-01"
END_DATE = date.today().isoformat()

SERIES_DICTIONARY_PATH = PROJECT_ROOT / "data" / "series_dictionary.csv"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
COMBINED_OUTPUT_PATH = RAW_DATA_DIR / "bcb_sgs_all_long.csv"

In [4]:
series_dictionary = pd.read_csv(SERIES_DICTIONARY_PATH)
verified_series = series_dictionary.loc[
    series_dictionary["verified"].astype(str).str.lower().eq("yes")
].copy()

verified_series[["name", "series_id", "frequency", "unit"]]

,name,series_id,frequency,unit
3,selic_monthly_annualized,4189,monthly,percent_annual
4,inflation_target,13521,annual,percent_annual
5,ipca,433,monthly,percent_monthly
6,ipca_12m,13522,monthly,percent_12_months
7,industrial_output_general,21859,monthly,index_2022_100
8,gdp_monthly_current_prices,4380,monthly,current_brl
10,exchange_rate_usd_sale_avg,3698,monthly,cmu_per_usd
11,credit_total_stock,20539,monthly,brl
12,credit_total_stock_firms,20540,monthly,brl
13,credit_total_stock_households,20541,monthly,brl


In [5]:
import pandas as pd

PANEL_FREQ = "MS"  # monthly start


def normalize_to_monthly(
    df: pd.DataFrame,
    *,
    name: str,
    frequency: str,
    start_date: str,
    end_date: str,
    aggregation: str | None = None,
) -> pd.DataFrame:
    """
    Convert raw SGS output to a monthly panel series.

    Expected input columns:
      date, value, series

    Rules:
      - monthly: align to monthly-start dates
      - daily: aggregate within month
      - annual: forward-fill annual value across months
    """

    monthly_index = pd.date_range(start=start_date, end=end_date, freq=PANEL_FREQ)

    if df.empty:
        return pd.DataFrame(
            {
                "date": monthly_index,
                "value": pd.NA,
                "series": name,
            }
        )

    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.sort_values("date").set_index("date")[["value"]]

    frequency = frequency.lower()

    if frequency == "monthly":
        # BCB monthly dates are usually already month-start.
        # This makes them explicit and safely aligns to the panel calendar.
        out.index = out.index.to_period("M").to_timestamp(how="start")
        out = out.groupby(level=0).last()

    elif frequency == "daily":
        # Default: monthly average for rates, exchange rates, indexes.
        # Use sum only for flow variables if explicitly needed.
        if aggregation is None:
            aggregation = "mean"

        if aggregation == "mean":
            out = out.resample("MS").mean()
        elif aggregation == "last":
            out = out.resample("MS").last()
        elif aggregation == "sum":
            out = out.resample("MS").sum()
        else:
            raise ValueError(f"Unsupported daily aggregation: {aggregation}")

    elif frequency == "annual":
        # Annual targets usually enter as the value applying to that year.
        out.index = out.index.to_period("Y").to_timestamp(how="start")
        out = out.groupby(level=0).last()
        out = out.reindex(monthly_index).ffill()

    else:
        raise ValueError(f"Unsupported frequency: {frequency}")

    out = (
        out.reindex(monthly_index)
           .rename_axis("date")
           .reset_index()
    )
    out["series"] = name

    return out[["date", "value", "series"]]

In [6]:
downloaded = []
summary_rows = []

for row in verified_series.itertuples(index=False):
    raw = fetch_sgs_series(
        series_id=row.series_id,
        start_date=START_DATE,
        end_date=END_DATE,
        name=row.name,
    )

    df = normalize_to_monthly(
        raw,
        name=row.name,
        frequency=row.frequency,
        start_date=START_DATE,
        end_date=END_DATE,
        aggregation=getattr(row, "aggregation", None),
    )

    output_path = RAW_DATA_DIR / f"{row.name}.csv"
    save_series_csv(df, output_path)
    downloaded.append(df)

    has_values = df["value"].notna().any()

    summary_rows.append(
        {
            "series": row.name,
            "series_id": row.series_id,
            "frequency": row.frequency,
            "observations": df["value"].notna().sum(),
            "first_date": df.loc[df["value"].notna(), "date"].min()
            if has_values
            else pd.NaT,
            "last_date": df.loc[df["value"].notna(), "date"].max()
            if has_values
            else pd.NaT,
        }
    )

if downloaded:
    combined = pd.concat(downloaded, ignore_index=True)
else:
    combined = pd.DataFrame(columns=["date", "value", "series"])

save_series_csv(combined, COMBINED_OUTPUT_PATH)

summary = pd.DataFrame(summary_rows)
summary["first_date"] = pd.to_datetime(summary["first_date"]).dt.date
summary["last_date"] = pd.to_datetime(summary["last_date"]).dt.date

print(summary.to_string(index=False))

                                              series  series_id frequency  observations first_date  last_date
                            selic_monthly_annualized       4189   monthly           186 2011-01-01 2026-06-01
                                    inflation_target      13521    annual           186 2011-01-01 2026-06-01
                                                ipca        433   monthly           184 2011-01-01 2026-04-01
                                            ipca_12m      13522   monthly           184 2011-01-01 2026-04-01
                           industrial_output_general      21859   monthly           183 2011-01-01 2026-03-01
                          gdp_monthly_current_prices       4380   monthly           184 2011-01-01 2026-04-01
                          exchange_rate_usd_sale_avg       3698   monthly           184 2011-01-01 2026-04-01
                                  credit_total_stock      20539   monthly           184 2011-01-01 2026-04-01
          